# 20.3 Örnek 1: CNN ile el yazısı rakam tanıma

*Makine Öğrenmesi Çalışma Rehberi · Bölüm 20: PyTorch ile Uygulamalı Örnekler*

**Nasıl kullanılır?** Hücreleri sırayla çalıştır (Shift + Enter) ya da menüden *Çalışma zamanı → Tümünü çalıştır*'ı seç. GPU gerekmez. Kodu değiştirip yeniden çalıştırmaktan çekinme. Köşeli parantezli numaralar ([76] gibi) PDF'teki kaynakçaya karşılık gelir.

### Veri seti

scikit-learn ile birlikte gelen küçük bir rakam veri setini kullanacağız; internetten indirme gerekmez. Veri seti Boğaziçi Üniversitesi'nde E. Alpaydın ve C. Kaynak tarafından hazırlanmıştır [77, 78]. 43 kişinin doldurduğu formlardaki rakamlar önce 32×32 piksel olarak taranmış, sonra her 4×4'lük bloktaki dolu pikseller sayılarak 8×8'lik görüntülere indirilmiştir. Bu yüzden her piksel 0 ile 16 arasında bir tamsayıdır. scikit-learn'deki sürüm, veri setinin 13 kişinin yazdığı test kısmıdır ve 1.797 görüntüden oluşur. (Ethem Alpaydın, pek çok üniversitede okutulan *Introduction to Machine Learning* ders kitabının da yazarıdır.)

*Şekil 20.1: Veri setinden her rakamın ilk örneği. Her görüntü 8×8 = 64 pikseldir; koyu pikseller yüksek değerlidir. (Şekil PDF'te.)*

### 1) Veri

In [ ]:
"""Örnek 1: CNN ile el yazısı rakam tanıma (PyTorch).

Veri seti scikit-learn ile birlikte gelir; internetten indirme gerekmez.
Kurulum:  pip install torch scikit-learn
"""
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)  # sonuçlar tekrarlanabilir olsun

In [ ]:
# 1) Veri: 1.797 adet 8×8 rakam görüntüsü, piksel değerleri 0–16
X, y = load_digits(return_X_y=True)
X = (X / 16.0).reshape(-1, 1, 8, 8)  # 0–1 aralığı; (örnek, kanal, yükseklik, genişlik)

# Önce test kümesini ayır, sonra kalan veriden doğrulama kümesini
X_gec, X_test, y_gec, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_egt, X_dog, y_egt, y_dog = train_test_split(
    X_gec, y_gec, test_size=0.2, stratify=y_gec, random_state=42)


def tensore_cevir(X, y):
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


X_egt, y_egt = tensore_cevir(X_egt, y_egt)
X_dog, y_dog = tensore_cevir(X_dog, y_dog)
X_test, y_test = tensore_cevir(X_test, y_test)

Görüntüler `nn.Conv2d`'nin beklediği (örnek, kanal, yükseklik, genişlik) biçimine getirilir; gri tonlu görüntülerde kanal sayısı 1'dir. Veri, Bölüm 6'daki gibi üçe bölünür: 1.149 eğitim, 288 doğrulama ve 360 test görüntüsü. `stratify`, her kümede rakamların oranının korunmasını sağlar. Etiketler sınıf numarası olduğu için `torch.long` (tamsayı) türünde tutulur; `nn.CrossEntropyLoss` bunu bekler.

### 2) Model

In [ ]:
# 2) Model: iki evrişim bloğu + sınıflandırıcı
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 1×8×8  → 16×8×8
    nn.ReLU(),
    nn.MaxPool2d(2),                              # 16×8×8 → 16×4×4
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16×4×4 → 32×4×4
    nn.ReLU(),
    nn.MaxPool2d(2),                              # 32×4×4 → 32×2×2
    nn.Flatten(),                                 # 32×2×2 → 128
    nn.Dropout(0.25),
    nn.Linear(128, 10),                           # 10 rakam için skor (logit)
)
print("Parametre sayısı:", sum(p.numel() for p in model.parameters()))

`nn.Sequential` katmanları yazıldıkları sırayla uygular. Her satırın sonundaki yorum, verinin o katmandan sonraki boyutunu gösterir. Son katmanda softmax yoktur: Model her rakam için ham bir skor (logit) üretir, softmax'ı kayıp fonksiyonu kendi içinde uygular.

**Tablo 20.2: Boyutlar ve parametre sayıları (kesin)**

| Katman | Çıktı boyutu | Parametre | Hesap (Bölüm 15) |
|---|---|---|---|
| Girdi | 1×8×8 | 0 | — |
| Conv2d 3×3, 16 filtre, dolgu 1 + ReLU | 16×8×8 | 160 | (3·3·1 + 1) · 16; boyut (8 − 3 + 2)/1 + 1 = 8 |
| MaxPool2d 2×2 | 16×4×4 | 0 | 8 / 2 |
| Conv2d 3×3, 32 filtre, dolgu 1 + ReLU | 32×4×4 | 4.640 | (3·3·16 + 1) · 32 |
| MaxPool2d 2×2 | 32×2×2 | 0 | 4 / 2 |
| Flatten, Dropout | 128 | 0 | 32 · 2 · 2 |
| Linear | 10 | 1.290 | 128 · 10 + 10 |
| **Toplam** |  | **6.090** |  |

Kodun ilk çıktısı olan parametre sayısı da 6090'dır. Bölüm 15'teki MNIST ağının 225.034 parametresiyle karşılaştır: Görüntüler küçük olduğu ve havuzlama boyutu hızla küçülttüğü için bu ağ çok daha hafiftir.

### 3) Kayıp, optimizasyon ve değerlendirme

In [ ]:
# 3) Kayıp fonksiyonu, optimizasyon ve mini-yığınlar
kayip_fn = nn.CrossEntropyLoss()  # softmax + çapraz entropi bir arada
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
yukleyici = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_egt, y_egt), batch_size=64, shuffle=True)


def dogruluk(X, y):
    model.eval()            # değerlendirme modu: dropout kapalı
    with torch.no_grad():   # türev kaydı tutma
        return (model(X).argmax(dim=1) == y).float().mean().item()

`DataLoader` eğitim verisini her epokta karıştırır ve 64'lük mini-yığınlar hâlinde verir; 1.149 örnek için bu, epok başına 18 adım demektir. `dogruluk` fonksiyonu modeli değerlendirme moduna alır (`model.eval()`: dropout kapanır) ve `torch.no_grad()` ile türev kaydını kapatır. Tahmin edilen sınıf, en yüksek skoru alan sınıftır (`argmax`).

### 4) Eğitim ve 5) Test

In [ ]:
# 4) Eğitim döngüsü
for epok in range(1, 31):
    model.train()           # eğitim modu: dropout açık
    toplam = 0.0
    for xb, yb in yukleyici:
        opt.zero_grad()                  # eski gradyanları sıfırla
        kayip = kayip_fn(model(xb), yb)  # ileri yayılım + kayıp
        kayip.backward()                 # geri yayılım
        opt.step()                       # ağırlıkları güncelle
        toplam += kayip.item() * len(xb)
    if epok % 5 == 0:
        print(f"Epok {epok:2d} | eğitim kaybı {toplam / len(X_egt):.3f} | "
              f"doğrulama doğruluğu {dogruluk(X_dog, y_dog):.3f}")

In [ ]:
# 5) Test kümesi: yalnızca en sonda ve bir kez
print(f"Test doğruluğu: {dogruluk(X_test, y_test):.3f}")

Her epokta önce `model.train()` ile dropout açılır, sonra 18 mini-yığının her biri için dört adımlık döngü çalışır. Beş epokta bir, ortalama eğitim kaybı ve doğrulama doğruluğu yazdırılır. Test kümesi yalnızca en sonda ve bir kez kullanılır (Bölüm 6.1).

**Beklenen çıktı** (PyTorch 2.14, işlemci, tohum 0)

```text
Parametre sayısı: 6090
Epok  5 | eğitim kaybı 0.431 | doğrulama doğruluğu 0.962
Epok 10 | eğitim kaybı 0.173 | doğrulama doğruluğu 0.976
Epok 15 | eğitim kaybı 0.105 | doğrulama doğruluğu 0.979
Epok 20 | eğitim kaybı 0.065 | doğrulama doğruluğu 0.986
Epok 25 | eğitim kaybı 0.048 | doğrulama doğruluğu 0.983
Epok 30 | eğitim kaybı 0.033 | doğrulama doğruluğu 0.990
Test doğruluğu: 0.981
```

> **NE GÖRMELİSİN? · Örnek 1**
>
> Aynı kod 10 farklı tohumla (0–9) çalıştırıldığında sonuçlar şu aralıklarda kaldı:
> - **Parametre sayısı:** 6090 (kesin).
> - **Epok 5:** eğitim kaybı 0,37–0,48; doğrulama doğruluğu %93–96.
> - **Epok 10:** eğitim kaybı 0,15–0,22; doğrulama doğruluğu %96–98.
> - **Epok 30:** eğitim kaybı 0,03–0,05; doğrulama doğruluğu %97,5–99,5.
> - **Test doğruluğu:** yaklaşık %98 (10 denemede %97,2 ile %98,6 arası).
> - **Karşılaştırma:** Aynı eğitim verisiyle eğitilen lojistik regresyon, aynı test kümesinde %96,1 doğruluk alır (kesin, scikit-learn ile hesaplandı).

*Şekil 20.2: Yukarıdaki çalıştırmanın (tohum 0) her epoktaki eğitim kaybı ve doğrulama doğruluğu. Kayıp, rastgele tahmin seviyesinden (ln 10 ≈ 2,30) hızla düşüyor; doğrulama doğruluğu ilk beş epokta %95'in üzerine çıkıyor. (Şekil PDF'te.)*

**İpucu:** Eğitime başlamadan önce kaybı bir kez yazdırırsan 2,30'a yakın bir değer görmelisin. On sınıfa eşit olasılık veren, henüz hiçbir şey öğrenmemiş bir modelin çapraz entropisi −ln(0,1) = ln(10) ≈ 2,30'dur (Bölüm 5.4). Bundan çok farklı bir başlangıç kaybı, kodda bir hata olduğunu gösterir.

### Kendin dene

**1.** CNN'deki filtre sayılarını 16 ve 32'den 8 ve 16'ya indir. Parametre sayısını önce Bölüm 15'teki formülle hesapla, sonra kodla kontrol et. İpucu: Düzleştirmeden sonraki boyut da değişir, `nn.Linear(128, 10)` satırını da güncellemelisin. (Cevap: 1.898.)

**2.** `nn.Dropout(0.25)` satırını sil. Eğitim kaybı ve doğrulama doğruluğu nasıl değişiyor? (Bölüm 7)

**3.** Öğrenme oranını 3e-3 yerine 3e-2 ve 3e-4 yap; kayıp eğrilerini karşılaştır (Bölüm 4.3).

**6.** (İleri) torchvision kütüphanesiyle gerçek MNIST veri setini (28×28) indir ve Bölüm 15'teki tabloda verilen CNN'i kur: `nn.Conv2d(1, 32, 3)`, `nn.MaxPool2d(2)`, `nn.Conv2d(32, 64, 3)`, `nn.MaxPool2d(2)`, `nn.Flatten()`, `nn.Linear(1600, 128)`, `nn.Linear(128, 10)` (aralara ReLU koymayı unutma). Parametre sayısı 225.034 çıkmalı.